# T33 - 信用债规模

## 第1章：项目概述与数据字典

---

### 项目概述

信用债规模项目专注于统计和分析债券市场的存续规模数据，包括不同类型（信用债、金融债、ABS等）、不同期限、不同评级、不同发行人类型的债券余额分布，为市场分析、信用评估和投资决策提供规模数据支持。

**核心功能**：
1. 从数据库提取原始债券数据
2. 分类、评级、久期补全
3. 多维度汇总信用债规模

---

### 项目定位

**信用债规模数据ETL处理系统**

| 阶段 | 输入 | 输出 | 说明 |
|------|------|------|------|
| 债券分类重建 | basicinfo_credit, basicinfo_finance, 债券新分类表 | bond.债券分类表 | 债券类型分类 |
| 评级补全 | marketinfo_*系列表 | bond.债券分类.评级字段 | 隐含评级补全 |
| 久期补全 | marketinfo_* + basicinfo_* | bond.marketinfo3 | 久期计算 |
| 多维度汇总 | marketinfo3 + 债券分类 + 辅助表 | bond.信用债规模 | 规模统计 |

### 数据流向图

```
基础信息表 → 债券分类表 → 市场信息表
     ↓              ↓              ↓
                 市场信息3表 → 信用债规模表
```

**表依赖关系**：
- `basicinfo_credit`, `basicinfo_finance`, `basicinfo_abs` → `债券分类`
- `marketinfo_*` (评级) → `债券分类`
- `marketinfo_*` + `basicinfo_*` → `marketinfo3`
- `marketinfo3` + `债券分类` + `辅助表` → `信用债规模`

### 数据字典

#### 1. 输入表

| 表名 | 数据库 | 说明 |
|------|--------|------|
| basicinfo_credit | bond | 信用债基础信息 |
| basicinfo_finance | bond | 金融债基础信息 |
| basicinfo_abs | bond | ABS基础信息 |
| marketinfo_credit | bond | 信用债市场信息（日度） |
| marketinfo_finance | bond | 金融债市场信息（日度） |
| marketinfo_abs | bond | ABS市场信息（日度） |
| basicinfo_xzqh_ct | bond | 城投行政区划映射 |
| basicinfo_industrytype1 | bond | 申万行业分类映射 |
| basicinfo_all | bond | 债券基础信息汇总表 |

#### 2. 中间表

| 表名 | 字段 | 说明 |
|------|------|------|
| marketinfo3 | DT, trade_code, ths_bond_balance_bond, 久期, 隐含评级 | 市场信息汇总表 |
| 债券分类 | trade_code, 大类, 子类, 评级 | 债券分类汇总表 |

#### 3. 输出表

| 表名 | 字段 | 说明 |
|------|------|------|
| 信用债规模 | DT, 分类, 子类, 分类方式, 余额 | 规模汇总表 |

### 分类维度说明

#### 债券类型分类

| 大类 | 子类分类依据 | 示例 |
|------|--------------|------|
| 信用债-城投 | 行政区域 | 江苏/南京 |
| 信用债-产业 | 申万行业 | 银行/非银金融 |
| 金融债 | 机构类型 | 银行/证券/保险 |
| ABS | 基础资产类型 | 消费贷ABS |

#### 期限分类

| 期限标签 | 久期范围 |
|----------|----------|
| 0.5年 | < 0.75年 |
| 1年 | 0.75 ~ 1.5年 |
| 2年 | 1.5 ~ 2.5年 |
| 3年 | 2.5 ~ 3.5年 |
| 4年 | 3.5 ~ 4.5年 |
| 5年 | 4.5 ~ 6年 |
| 7年 | 6 ~ 8年 |
| 10年 | >= 8年 |

#### 评级分类

| 评级级别 | 说明 |
|----------|------|
| AAA | 最高评级 |
| AA+ | 高评级 |
| AA | 中高评级 |
| AA- | 中评级 |
| A | 中低评级 |
| 其他 | 其他评级 |
| 无评级 | 无评级债券 |

### 核心字段说明

#### 债券基础信息表关键字段

| 字段名 | 类型 | 说明 |
|--------|------|------|
| trade_code | VARCHAR | 债券代码 |
| bond_name | VARCHAR | 债券名称 |
| issuer | VARCHAR | 发行人 |
| issue_date | DATE | 发行日期 |
| maturity_date | DATE | 到期日期 |
| coupon_rate | DECIMAL | 票面利率 |
| ths_bond_balance_bond | DECIMAL | 债券余额（亿元） |
| ths_cb_market_implicit_rating_bond | VARCHAR | 隐含评级 |
| ths_is_city_invest_debt_yy_bond | VARCHAR | 是否城投债 |
| ths_bond_maturity_theory_bond | VARCHAR | 理论期限（永续债格式） |
| ths_object_the_sw_bond | VARCHAR | 申万行业 |
| ths_org_type_bond | VARCHAR | 机构类型 |

#### 市场信息表关键字段

| 字段名 | 类型 | 说明 |
|--------|------|------|
| dt | DATE | 交易日期 |
| trade_code | VARCHAR | 债券代码 |
| close | DECIMAL | 收盘价 |
| ths_evaluate_modified_dur_cb_bond_exercise | DECIMAL | 修正久期（行权） |
| ths_evaluate_interest_durcb_bond_exercise | DECIMAL | 利息久期（行权） |
| ths_bond_balance_bond | DECIMAL | 债券余额 |
| ths_cb_market_implicit_rating_bond | VARCHAR | 隐含评级 |

### 规模统计分类方式

| 分类方式 | 分类字段 | 子类字段 | 说明 |
|---------|---------|---------|------|
| 大类 | '全部'或大类名称 | '' | 按债券大类汇总 |
| 省 | '城投' | 省名 | 城投债按省份汇总 |
| 市 | '城投' | 市名 | 城投债按城市汇总 |
| 申万一级 | '产业' | 申万一级行业 | 产业债按一级行业汇总 |
| 申万行业 | '产业' | 申万行业名称 | 产业债按具体行业汇总 |
| 一级分类 | '产业' | 一级分类 | 产业债按一级分类汇总 |
| 二级分类 | '产业' | 二级分类 | 产业债按二级分类汇总 |
| 金融机构 | '金融' | 机构类型 | 金融债按机构类型汇总 |
| 资产 | 'ABS' | ABS类型 | ABS按基础资产类型汇总 |
| 评级 | 大类或'全部' | 评级等级 | 按评级汇总 |
| 久期 | 大类或'全部' | 久期标签 | 按久期段汇总 |

In [ ]:
# 环境验证
import sys
print(f"Python版本: {sys.version}")
print(f"Python路径: {sys.executable}")

In [ ]:
# 验证必要的库
import pandas as pd
import sqlalchemy
from datetime import datetime
print("核心库导入成功")
print(f"Pandas版本: {pd.__version__}")
print(f"SQLAlchemy版本: {sqlalchemy.__version__}")

---

### 项目执行流程

```
1. Truncate bond.债券分类
2. INSERT债券分类（信用债）
3. INSERT债券分类（金融债）
4. INSERT债券分类（ABS）
5. 循环3个表更新评级
6. 循环3个表插入marketinfo3（久期存在）
7. 循环3个表计算并更新久期缺失记录
8. 循环15个INSERT语句插入信用债规模
```

---

**下一章**: [2_环境配置与初始化](2_环境配置与初始化.ipynb)